In [25]:
import os
import cv2
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Reshape, Bidirectional, LSTM, Dense, Activation, BatchNormalization, Dropout, Lambda
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, LearningRateScheduler
from PIL import Image
import glob

In [ ]:

alphabets = u" !\"%'()*+,-./0123456789:;=?ABCDEFGHIJKLMNOPQRSTUVWXYZ_abcdefghijklmnopqrstuvwxyz{}¤°²ÀÉàâçèéêëîôùûœ€"

max_str_len = 128


num_of_characters = len(alphabets) + 1

num_of_timestamps = 128

In [ ]:
def resize_or_pad_image(image, target_height=64, target_width=512):
    """
    Resizes an image to a target width and height.
    """
  
    if len(image.shape) == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    
    image = cv2.resize(image, (target_width, target_height))
    return image

In [ ]:
def augment_image(image):
    """
    Applies various data augmentation techniques to an image randomly.
    """
  
    if np.random.rand() < 0.3: 
        kernel_size = tuple(np.random.randint(1, 3, 2) * 2 + 1)
        image = cv2.GaussianBlur(image, kernel_size, 0)

    if np.random.rand() < 0.3: 
        brightness_factor = np.random.uniform(0.8, 1.2)
        image = np.clip(image * brightness_factor, 0, 255).astype(np.uint8)

   
    if np.random.rand() < 0.3: 
        contrast_factor = np.random.uniform(0.7, 1.3)
        image = np.clip((image - 127.5) * contrast_factor + 127.5, 0, 255).astype(np.uint8)

    if np.random.rand() < 0.3: 
        rows, cols, _ = image.shape
        shear_angle_x = np.random.uniform(-0.03, 0.03) 
        shear_matrix_x = np.float32([[1, shear_angle_x, 0], [0, 1, 0]])
        sheared_image = cv2.warpAffine(image.copy(), shear_matrix_x, (cols, rows),
                                       borderMode=cv2.BORDER_REPLICATE)
        image = sheared_image

   
    if np.random.rand() < 0.2: 
        angle = np.random.uniform(-2, 2) 
        rows, cols, _ = image.shape
        M = cv2.getRotationMatrix2D(((cols-1)/2.0,(rows-1)/2.0),angle,1)
        image = cv2.warpAffine(image,M,(cols,rows), borderMode=cv2.BORDER_REPLICATE)

    return image


In [33]:
def preprocess_image_for_model(img, augment=False):
    """
    Preprocesses a single image for the model.
    Applies resizing, optional augmentation, rotation, and normalization.
    """
    # Ensure image is grayscale for initial processing
    if len(img.shape) == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # If augmenting, convert to 3 channels first for augmentation functions
    # then convert back to grayscale before final rotation and normalization.
    if augment:
        img_bgr = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        img_augmented_bgr = augment_image(img_bgr)
        img = cv2.cvtColor(img_augmented_bgr, cv2.COLOR_BGR2GRAY) # Convert back to grayscale
    
    # Resize/pad to target dimensions (512, 64)
    final_image = cv2.resize(img, (512, 64))

    # Rotate 90 degrees clockwise (as done in your original notebook for HTR)
    final_image = cv2.rotate(final_image, cv2.ROTATE_90_CLOCKWISE)

    # Normalize pixel values to [0, 1]
    final_image = final_image.astype('float32') / 255.0

    # Expand dimensions to (height, width, channels=1) for Keras CNN input
    return np.expand_dims(final_image, axis=-1)


In [34]:
def label_to_num(label):
    """Converts a text label to a sequence of numerical indices."""
    label_num = []
    for ch in label:
        ind = alphabets.find(ch)
        if ind != -1:
            label_num.append(ind)
        else:
            # Map unknown characters to 0 (or a dedicated UNK token index).
            # This is critical for handling characters not in your defined alphabet.
            label_num.append(0) # Assuming 0 is either a valid character or a blank/UNK index.
    return np.array(label_num)


In [35]:
def num_to_label(num):
    """Converts a sequence of numerical indices back to a text label."""
    ret = ""
    for ch in num:
        if ch == -1 or ch == len(alphabets):  # -1 is padding, last index is blank in CTC.
            break
        else:
            ret += alphabets[ch]
    return ret

In [36]:
@keras.utils.register_keras_serializable()
def ctc_lambda_func(args):
    """
    Lambda layer to compute the CTC loss.
    y_pred: Predicted probabilities from the model's output layer (softmax).
    labels: True labels (encoded text sequences).
    input_length: Length of the predicted sequences (from CNN output).
    label_length: Length of the true label sequences.
    """
    y_pred, labels, input_length, label_length = args
    # The 2 is critical here since the first couple outputs of the RNN
    # tend to be garbage or blank due to CTC's nature.
    y_pred = y_pred[:, 2:, :]
    
    # IMPORTANT: Adjust input_length to match the sliced y_pred
    input_length = input_length - 2
    
    return tf.keras.backend.ctc_batch_cost(labels, y_pred, input_length, label_length)


In [37]:

# Fixed CTCCallback class
class CTCCallback(keras.callbacks.Callback):
    """
    Custom Keras Callback to compute and display Character Error Rate (CER)
    during training.
    """
    def __init__(self, validation_data_generator, prediction_model):
        super().__init__()
        self.validation_data_generator = validation_data_generator
        self.prediction_model = prediction_model

    def on_epoch_end(self, epoch, logs=None):
        all_original_texts = []
        all_decoded_preds = []

        # Iterate through batches of validation data
        for i in range(len(self.validation_data_generator)):
            batch_inputs, _ = self.validation_data_generator[i]
            batch_images = batch_inputs['image']
            batch_true_labels_padded = batch_inputs['labels'] # Padded numerical labels

            # Get predictions for the batch
            batch_preds = self.prediction_model.predict(batch_images, verbose=0)
            
            # Decode batch predictions
            decoded_batch_preds = self.decode_predictions(batch_preds)
            all_decoded_preds.extend(decoded_batch_preds)

            # Convert true labels (padded numerical) to text for comparison
            for j in range(batch_true_labels_padded.shape[0]):
                all_original_texts.append(num_to_label(batch_true_labels_padded[j]))

        # Calculate Levenshtein distance (edit distance)
        def levenshtein_distance(s1, s2):
            if len(s1) < len(s2):
                return levenshtein_distance(s2, s1)
            if len(s2) == 0:
                return len(s1)
            previous_row = range(len(s2) + 1)
            for i, c1 in enumerate(s1):
                current_row = [i + 1]
                for j, c2 in enumerate(s2):
                    insertions = previous_row[j + 1] + 1
                    deletions = current_row[j] + 1
                    substitutions = previous_row[j] + (c1 != c2)
                    current_row.append(min(insertions, deletions, substitutions))
                previous_row = current_row
            return previous_row[-1]

        total_cer = 0
        num_samples = len(all_original_texts)
        if num_samples == 0:
            avg_cer = 0.0
        else:
            for i in range(num_samples):
                true_text = all_original_texts[i]
                pred_text = all_decoded_preds[i]

                if len(true_text) == 0:
                    cer = 1.0 if len(pred_text) > 0 else 0.0
                else:
                    cer = levenshtein_distance(true_text, pred_text) / len(true_text)
                total_cer += cer
            avg_cer = (total_cer / num_samples) * 100

        print(f"Epoch {epoch + 1}: Val CER = {avg_cer:.2f}%")
        logs['val_cer'] = avg_cer

    def decode_predictions(self, pred):
        """Helper to decode raw model predictions to text within the callback."""
        input_len = np.ones(pred.shape[0]) * pred.shape[1]
        
        # FIXED: Remove merge_repeated parameter which is not supported in newer Keras versions
        try:
            # Try with merge_repeated (for older versions)
            results = K.ctc_decode(pred, input_length=input_len, greedy=True, merge_repeated=True)[0][0]
        except TypeError:
            # Fall back to without merge_repeated (for newer versions)
            results = K.ctc_decode(pred, input_length=input_len, greedy=True)[0][0]
        
        output_texts = []
        for res in results.numpy():
            output_texts.append(num_to_label(res))
        return output_texts

In [39]:
def load_data(images_dir, labels_dir):
    """Load and associate images and labels."""
    data = []
    image_files = glob.glob(os.path.join(images_dir, "*.jpg"))
    for img_path in image_files:
        filename = os.path.splitext(os.path.basename(img_path))[0]
        label_path = os.path.join(labels_dir, f"{filename}.txt")
        if os.path.exists(label_path):
            try:
                with open(label_path, 'r', encoding='utf-8') as f:
                    label_text = f.read().strip()
                if label_text:
                    data.append({
                        'image_path': img_path,
                        'label_path': label_path,
                        'text': label_text,
                        'filename': filename
                    })
            except Exception as e:
                print(f"Erreur lecture {label_path}: {e}")
    print(f"Loaded {len(data)} samples from {images_dir}")
    return data

In [46]:
def build_vocab(data):
    """Builds the vocabulary of characters."""
    chars = set()
    for item in data:
        chars.update(item['text'])
    
    # Add special tokens if they are not already in your data
    # These are not directly used by CTC for prediction but can be in vocab if needed.
    chars.add('<PAD>')
    chars.add('<SOS>')
    chars.add('<EOS>')
    chars.add('<UNK>')
    
    chars = sorted(list(chars))
    char_to_idx = {char: idx for idx, char in enumerate(chars)}
    idx_to_char = {idx: char for char, idx in char_to_idx.items()}
    
    print(f"Vocabulary size: {len(chars)}")
    print(f"Characters: {''.join(sorted([c for c in chars if c not in ['<PAD>', '<SOS>', '<EOS>', '<UNK>']]))}\n")
    
    return char_to_idx, idx_to_char

In [47]:
def load_kaggle_dataset(dataset_path="/kaggle/input/rimes1/dl"):
    """
    Loads the dataset from Kaggle structure.
    """
    train_images_dir = os.path.join(dataset_path, "photos", "train")
    train_labels_dir = os.path.join(dataset_path, "label", "train")
    eval_images_dir = os.path.join(dataset_path, "photos", "eval")
    eval_labels_dir = os.path.join(dataset_path, "label", "eval")
    
    for path in [train_images_dir, train_labels_dir, eval_images_dir, eval_labels_dir]:
        if not os.path.exists(path):
            print(f"Attention: {path} n'existe pas")
    
    train_data = load_data(train_images_dir, train_labels_dir)
    eval_data = load_data(eval_images_dir, eval_labels_dir)
    
    # Build vocabulary from training data
    # (Note: The global `alphabets` should ideally be derived from this `build_vocab` for consistency)
    char_to_idx, idx_to_char = build_vocab(train_data)
    
    return train_data, eval_data, char_to_idx, idx_to_char


In [49]:
class CustomKerasDataGenerator(tf.keras.utils.Sequence):
    """
    A Keras-compatible data generator that loads images, applies preprocessing
    and augmentation, and prepares data for CTC loss.
    """
    def __init__(self, data, char_to_idx, max_str_len, num_of_timestamps, batch_size, augment=False, shuffle=True, **kwargs):
        # Call the parent class constructor first to fix the PyDataset warning
        super().__init__(**kwargs)
        
        self.data = data
        self.char_to_idx = char_to_idx
        self.max_str_len = max_str_len
        self.num_of_timestamps = num_of_timestamps
        self.batch_size = batch_size
        self.augment = augment
        self.shuffle = shuffle
        self.on_epoch_end() # Initialize indices

    def __len__(self):
        """Returns the number of batches per epoch."""
        return int(np.floor(len(self.data) / self.batch_size))

    def __getitem__(self, index):
        """Generates one batch of data."""
        indices = self.indices[index * self.batch_size:(index + 1) * self.batch_size]
        batch_data = [self.data[k] for k in indices]
        return self.__data_generation(batch_data)

    def on_epoch_end(self):
        """Updates indices after each epoch (for shuffling)."""
        self.indices = np.arange(len(self.data))
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __data_generation(self, batch_data):
        """
        Generates data containing batch_size samples.
        X (images), Y (labels), input_length, label_length
        """
        batch_images = []
        batch_labels = []
        batch_input_length = []
        batch_label_length = []

        for item in batch_data:
            try:
                img = np.array(Image.open(item['image_path']).convert('L')) # Convert to grayscale
            except Exception as e:
                print(f"Error loading image {item['image_path']}: {e}. Skipping.")
                continue

            processed_img = preprocess_image_for_model(img, augment=self.augment)
            batch_images.append(processed_img)

            encoded_label = label_to_num(item['text'])
            
            # Pad labels to max_str_len with -1 (CTC padding token)
            # FIXED: Use int64 instead of float32 for labels
            padded_label = np.full((self.max_str_len,), -1, dtype=np.int64)
            padded_label[0:len(encoded_label)] = encoded_label
            batch_labels.append(padded_label)

            # FIXED: Input length for CTC accounting for the 2 timesteps removed in ctc_lambda_func
            batch_input_length.append(self.num_of_timestamps - 2)  # Changed from self.num_of_timestamps

            # Label length for CTC (actual length of the true label)
            batch_label_length.append(len(encoded_label))

        # Convert lists to numpy arrays
        X = np.array(batch_images)
        y_labels = np.array(batch_labels)
        y_input_length = np.array(batch_input_length, dtype=np.int64).reshape(-1, 1)
        y_label_length = np.array(batch_label_length, dtype=np.int64).reshape(-1, 1)

        # Keras model expects inputs as a dictionary and outputs as a dummy value
        inputs = {
            'image': X,
            'labels': y_labels,
            'input_length': y_input_length,
            'label_length': y_label_length
        }
        outputs = np.zeros(len(batch_images)) # Dummy output for the loss function

        return inputs, outputs

In [50]:
def build_htr_model_from_notebook(input_shape=(512, 64, 1), num_classes=num_of_characters):
    """
    Builds the Hand-Written Text Recognition (HTR) model based on the provided notebook's CNN structure.
    This model matches the convolutional and recurrent layers from your `english-htr.ipynb`.
    """
    input_data = Input(shape=input_shape, name='image') # Renamed to 'image' for generator compatibility
    labels = Input(name='labels', shape=[max_str_len], dtype='float32')
    input_length = Input(name='input_length', shape=[1], dtype='int64')
    label_length = Input(name='label_length', shape=[1], dtype='int64')

    # CNN layers (from your notebook)
    inner = Conv2D(32, (3, 3), padding='same', name='conv1', kernel_initializer='he_normal')(input_data)
    inner = BatchNormalization()(inner)
    inner = Activation('relu')(inner)
    inner = MaxPooling2D(pool_size=(2, 2), name='max1')(inner)

    inner = Conv2D(64, (3, 3), padding='same', name='conv2', kernel_initializer='he_normal')(inner)
    inner = BatchNormalization()(inner)
    inner = Activation('relu')(inner)
    inner = MaxPooling2D(pool_size=(2, 2), name='max2')(inner)
    inner = Dropout(0.3)(inner)

    inner = Conv2D(128, (3, 3), padding='same', name='conv3', kernel_initializer='he_normal')(inner)
    inner = BatchNormalization()(inner)
    inner = Activation('relu')(inner)
    inner = Conv2D(128, (3, 3), padding='same', name='conv4', kernel_initializer='he_normal')(inner)
    inner = BatchNormalization()(inner)
    inner = Activation('relu')(inner)
    inner = MaxPooling2D(pool_size=(1, 2), name='max3')(inner) # Keeps height (sequence length) at 128
    inner = Dropout(0.4)(inner)
    inner = BatchNormalization()(inner)

    # Reshape for RNN
    # K.int_shape(inner) at this point should be (None, 128, 8, 128)
    inner = Reshape(target_shape=(num_of_timestamps, K.int_shape(inner)[2] * K.int_shape(inner)[3]), name='reshape')(inner)
    inner = Dense(256, activation='relu', kernel_initializer='he_normal', name='dense1')(inner)

    # BiLSTM layers
    lstm_1 = Bidirectional(LSTM(512, return_sequences=True, dropout=0.2, recurrent_dropout=0.1), name='lstm1')(inner)
    lstm_2 = Bidirectional(LSTM(512, return_sequences=True, dropout=0.2, recurrent_dropout=0.1), name='lstm2')(lstm_1)

    # Output layer
    output_logits = Dense(num_classes, kernel_initializer='he_normal', name='dense2')(lstm_2)
    y_pred = Activation('softmax', name='softmax_output')(output_logits)

    # CTC Lambda layer for loss
    ctc_loss = Lambda(ctc_lambda_func, output_shape=(1,), name='ctc_loss')(
        [y_pred, labels, input_length, label_length]
    )

    # Training model
    model_for_training = Model(
        inputs=[input_data, labels, input_length, label_length],
        outputs=ctc_loss,
        name="htr_training_model"
    )
    optimizer = Adam(learning_rate=0.0001)
    model_for_training.compile(loss={'ctc_loss': lambda y_true, y_pred: y_pred}, optimizer=optimizer)

    # Prediction model (for inference)
    prediction_model = Model(
        inputs=input_data,
        outputs=y_pred,
        name="htr_prediction_model"
    )

    return model_for_training, prediction_model


In [51]:
    print("Starting HTR Model Training Script...")

    # Load your dataset
    train_data, eval_data, char_to_idx, idx_to_char = load_kaggle_dataset("/kaggle/input/rimes1/dl")

    # Display a check of the derived global variables
    print(f"\n--- Derived Global Variables ---")
    print(f"Alphabet: {alphabets}")
    print(f"Alphabet length: {len(alphabets)}")
    print(f"Num of characters (vocab size + blank): {num_of_characters}")
    print(f"Max string length for padding: {max_str_len}")
    print(f"Num of timestamps (CNN output sequence length): {num_of_timestamps}")
    print("-----------------------------\n")

    # Initialize data generators
    batch_size = 32
    train_generator = CustomKerasDataGenerator(
        data=train_data,
        char_to_idx=char_to_idx,
        max_str_len=max_str_len,
        num_of_timestamps=num_of_timestamps,
        batch_size=batch_size,
        augment=True, # Apply data augmentation for training
        shuffle=True
    )

    validation_generator = CustomKerasDataGenerator(
        data=eval_data,
        char_to_idx=char_to_idx,
        max_str_len=max_str_len,
        num_of_timestamps=num_of_timestamps,
        batch_size=batch_size,
        augment=False, # No augmentation for validation
        shuffle=False
    )

    # Build the model
    # This uses the CNN-BiLSTM structure from your `english-htr.ipynb`.
    # It takes grayscale images as input and produces 128 timestamps,
    # which is compatible with `max_str_len=128`.
    model_for_training, prediction_model = build_htr_model_from_notebook(
        input_shape=(512, 64, 1), # Input shape for the CNN (height, width, channels)
        num_classes=num_of_characters
    )

    print("\n--- Model Summary (Training Model) ---")
    model_for_training.summary()
    print("\n--- Model Summary (Prediction Model) ---")
    prediction_model.summary()

    # Callbacks for training
    
    
    

Starting HTR Model Training Script...
Loaded 11326 samples from /kaggle/input/rimes1/dl/photos/train
Loaded 778 samples from /kaggle/input/rimes1/dl/photos/eval
Vocabulary size: 104
Characters:  !"%'()*+,-./0123456789:;=?ABCDEFGHIJKLMNOPQRSTUVWXYZ_abcdefghijklmnopqrstuvwxyz{}¤°²ÀÉàâçèéêëîôùûœ€


--- Derived Global Variables ---
Alphabet:  !"%'()*+,-./0123456789:;=?ABCDEFGHIJKLMNOPQRSTUVWXYZ_abcdefghijklmnopqrstuvwxyz{}¤°²ÀÉàâçèéêëîôùûœ€
Alphabet length: 100
Num of characters (vocab size + blank): 101
Max string length for padding: 128
Num of timestamps (CNN output sequence length): 128
-----------------------------


--- Model Summary (Training Model) ---


Model: "htr_training_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ image (InputLayer)        │ (None, 512, 64, 1)     │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1 (Conv2D)            │ (None, 512, 64, 32)    │            320 │ image[0][0]            │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_10    │ (None, 512, 64, 32)    │            128 │ conv1[0][0]            │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation_8 (Activation) │ (None, 512, 64, 32)    │              0 │ batch_normalization_1… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ max1 (MaxPooling2D)       │ (None, 256, 32, 32)    │              0 │ activation_8[0][0]     │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2 (Conv2D)            │ (None, 256, 32, 64)    │         18,496 │ max1[0][0]             │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_11    │ (None, 256, 32, 64)    │            256 │ conv2[0][0]            │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation_9 (Activation) │ (None, 256, 32, 64)    │              0 │ batch_normalization_1… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ max2 (MaxPooling2D)       │ (None, 128, 16, 64)    │              0 │ activation_9[0][0]     │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_4 (Dropout)       │ (None, 128, 16, 64)    │              0 │ max2[0][0]             │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv3 (Conv2D)            │ (None, 128, 16, 128)   │         73,856 │ dropout_4[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_12    │ (None, 128, 16, 128)   │            512 │ conv3[0][0]            │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation_10             │ (None, 128, 16, 128)   │              0 │ batch_normalization_1… │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv4 (Conv2D)            │ (None, 128, 16, 128)   │        147,584 │ activation_10[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_13    │ (None, 128, 16, 128)   │            512 │ conv4[0][0]            │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation_11             │ (None, 128, 16, 128)   │              0 │ batch_normalization_1… │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ max3 (MaxPooling2D)  

 Total params: 10,053,477 (38.35 MB)

 Trainable params: 10,052,517 (38.35 MB)

...

Model: "htr_prediction_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ image (InputLayer)                   │ (None, 512, 64, 1)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1 (Conv2D)                       │ (None, 512, 64, 32)         │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_10               │ (None, 512, 64, 32)         │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_8 (Activation)            │ (None, 512, 64, 32)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max1 (MaxPooling2D)                  │ (None, 256, 32, 32)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2 (Conv2D)                       │ (None, 256, 32, 64)         │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_11               │ (None, 256, 32, 64)         │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_9 (Activation)            │ (None, 256, 32, 64)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max2 (MaxPooling2D)                  │ (None, 128, 16, 64)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_4 (Dropout)                  │ (None, 128, 16, 64)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv3 (Conv2D)                       │ (None, 128, 16, 128)        │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_12               │ (None, 128, 16, 128)        │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_10 (Activation)           │ (None, 128, 16, 128)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv4 (Conv2D)                       │ (None, 128, 16, 128)        │         147,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_13               │ (None, 128, 16, 128)        │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_11 (Activation)           │ (None, 128, 16, 128)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max3 (MaxPooling2D)                  │ (None, 128, 8, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_5 (Dropout)                  │ (None, 128, 8, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_14               │ (None, 128, 8, 128)         │             512 │
│ (BatchNormalization)                 │                             │              

 Total params: 10,053,477 (38.35 MB)

 Trainable params: 10,052,517 (38.35 MB)

 Non-trainable params: 960 (3.75 KB)

In [52]:
# Fix 1: Change the checkpoint filepath to end with .weights.h5
checkpoint_filepath = 'htr_model_best.weights.h5'  # Changed from 'htr_model_best.h5'
model_checkpoint_callback = ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    monitor='val_loss',
    mode='min',
    save_best_only=True,
    verbose=1
)

early_stopping_callback = EarlyStopping(
    monitor='val_loss',
    patience=15,  # Increased patience for longer training
    restore_best_weights=True,
    verbose=1
)

reduce_lr_callback = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=8,  # Increased patience
    min_lr=0.00001,
    verbose=1
)

# Custom CTC callback for CER calculation
ctc_callback = CTCCallback(validation_generator, prediction_model)

callbacks = [model_checkpoint_callback, early_stopping_callback, reduce_lr_callback, ctc_callback]

# Fix 2: Increase epochs as requested
epochs = 50  # Increased from 50 to 100 epochs
print(f"\nStarting model training for {epochs} epochs...")
history = model_for_training.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=epochs,
    callbacks=callbacks,
    verbose=1
)

print("\nTraining complete!")
print(f"Best model weights saved to: {checkpoint_filepath}")

# To load the weights later for inference:
# prediction_model.load_weights(checkpoint_filepath)


Starting model training for 50 epochs...
Epoch 1/50


E0000 00:00:1748982160.262949      35 meta_optimizer.cc:966] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/htr_training_model_1/dropout_4_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


353/353 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 167.1957
Epoch 1: val_loss improved from inf to 130.87343, saving model to htr_model_best.weights.h5
Epoch 1: Val CER = 91.20%
353/353 ━━━━━━━━━━━━━━━━━━━━ 523s 1s/step - loss: 167.1315 - val_loss: 130.8734 - learning_rate: 1.0000e-04 - val_cer: 91.2005
Epoch 2/50
353/353 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 113.0766
Epoch 2: val_loss improved from 130.87343 to 93.86501, saving model to htr_model_best.weights.h5
Epoch 2: Val CER = 69.75%
353/353 ━━━━━━━━━━━━━━━━━━━━ 497s 1s/step - loss: 113.0562 - val_loss: 93.8650 - learning_rate: 1.0000e-04 - val_cer: 69.7527
Epoch 3/50
353/353 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 88.1103
Epoch 3: val_loss improved from 93.86501 to 71.37900, saving model to htr_model_best.weights.h5
Epoch 3: Val CER = 48.30%
353/353 ━━━━━━━━━━━━━━━━━━━━ 497s 1s/step - loss: 88.0973 - val_loss: 71.3790 - learning_rate: 1.0000e-04 - val_cer: 48.2957
Epoch 4/50
353/353 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 69.6100


In [54]:
import pickle
import json

# 1. Save the complete prediction model (recommended for deployment)
print("Saving complete prediction model...")
prediction_model.save('/kaggle/working/htr_prediction_model.h5')
print("Complete model saved as '/kaggle/working/htr_prediction_model.h5'")

# 2. Save model weights separately (you're already doing this)
print("Saving model weights...")
prediction_model.save_weights('/kaggle/working/htr_model_weights.weights.h5')
print("Model weights saved as '/kaggle/working/htr_model_weights.weights.h5'")

# 3. Save the alphabet and configuration needed for deployment
model_config = {
    'alphabets': alphabets,
    'max_str_len': max_str_len,
    'num_of_characters': num_of_characters,
    'num_of_timestamps': num_of_timestamps,
    'input_shape': (512, 64, 1)
}

with open('/kaggle/working/model_config.json', 'w', encoding='utf-8') as f:
    json.dump(model_config, f, ensure_ascii=False, indent=2)
print("Model configuration saved as '/kaggle/working/model_config.json'")

# 4. Save character mappings if using build_vocab approach
try:
    char_mappings = {
        'char_to_idx': char_to_idx,
        'idx_to_char': idx_to_char
    }
    with open('/kaggle/working/char_mappings.pkl', 'wb') as f:
        pickle.dump(char_mappings, f)
    print("Character mappings saved as '/kaggle/working/char_mappings.pkl'")
except:
    print("Character mappings not found - using global alphabets variable")

print("\n" + "="*50)
print("MODEL SAVING COMPLETE!")
print("Files created in /kaggle/working/ for download:")
print("- htr_prediction_model.h5 (complete model)")
print("- htr_model_weights.weights.h5 (weights only)")
print("- model_config.json (configuration)")
print("- char_mappings.pkl (optional character mappings)")
print("- flask_app.py (Flask deployment code)")
print("- requirements.txt (Python dependencies)")
print("\nTo download: Go to Data tab → Output → Download all files")
print("="*50)

Saving complete prediction model...
Complete model saved as '/kaggle/working/htr_prediction_model.h5'
Saving model weights...
Model weights saved as '/kaggle/working/htr_model_weights.weights.h5'
Model configuration saved as '/kaggle/working/model_config.json'
Character mappings saved as '/kaggle/working/char_mappings.pkl'

MODEL SAVING COMPLETE!
Files created in /kaggle/working/ for download:
- htr_prediction_model.h5 (complete model)
- htr_model_weights.weights.h5 (weights only)
- model_config.json (configuration)
- char_mappings.pkl (optional character mappings)
- flask_app.py (Flask deployment code)
- requirements.txt (Python dependencies)

To download: Go to Data tab → Output → Download all files


In [55]:
def calculate_accuracy_from_notebook(validation_generator, prediction_model, num_batches=None):
    """
    Calcule l'accuracy en utilisant exactement les mêmes fonctions que votre notebook.
    Basé sur votre CTCCallback et vos fonctions existantes.
    """
    all_original_texts = []
    all_decoded_preds = []
    
    # Nombre de batches à évaluer
    batches_to_process = num_batches if num_batches else len(validation_generator)
    
    print(f"Calcul de l'accuracy sur {batches_to_process} batches...")
    
    # Itération sur les batches de données de validation (même logique que votre CTCCallback)
    for i in range(batches_to_process):
        if i >= len(validation_generator):
            break
            
        batch_inputs, _ = validation_generator[i]
        batch_images = batch_inputs['image']
        batch_true_labels_padded = batch_inputs['labels']  # Labels numériques paddés

        # Obtenir les prédictions pour le batch (même que votre CTCCallback)
        batch_preds = prediction_model.predict(batch_images, verbose=0)
        
        # Décoder les prédictions (même logique que votre CTCCallback)
        decoded_batch_preds = decode_predictions_notebook_style(batch_preds)
        all_decoded_preds.extend(decoded_batch_preds)

        # Convertir les vrais labels en texte (même que votre CTCCallback)
        for j in range(batch_true_labels_padded.shape[0]):
            original_text = num_to_label(batch_true_labels_padded[j])
            all_original_texts.append(original_text)
        
        if (i + 1) % 10 == 0:
            print(f"Traité {i + 1}/{batches_to_process} batches")

    # Calcul de la distance de Levenshtein (même fonction que votre CTCCallback)
    def levenshtein_distance(s1, s2):
        if len(s1) < len(s2):
            return levenshtein_distance(s2, s1)
        if len(s2) == 0:
            return len(s1)
        previous_row = range(len(s2) + 1)
        for i, c1 in enumerate(s1):
            current_row = [i + 1]
            for j, c2 in enumerate(s2):
                insertions = previous_row[j + 1] + 1
                deletions = current_row[j] + 1
                substitutions = previous_row[j] + (c1 != c2)
                current_row.append(min(insertions, deletions, substitutions))
            previous_row = current_row
        return previous_row[-1]

    # Calculs des métriques (même logique que votre CTCCallback)
    total_cer = 0
    total_characters = 0
    exact_matches = 0
    total_samples = len(all_original_texts)
    
    for i in range(total_samples):
        true_text = all_original_texts[i]
        pred_text = all_decoded_preds[i]

        # Calcul CER (même que votre CTCCallback)
        if len(true_text) == 0:
            cer = 1.0 if len(pred_text) > 0 else 0.0
        else:
            cer = levenshtein_distance(true_text, pred_text) / len(true_text)
        
        total_cer += cer
        total_characters += len(true_text)
        
        # Comptage des correspondances exactes
        if true_text == pred_text:
            exact_matches += 1

    # Métriques finales
    avg_cer = (total_cer / total_samples) * 100 if total_samples > 0 else 0.0
    character_accuracy = 100 - avg_cer  # Accuracy = 100 - CER
    sequence_accuracy = (exact_matches / total_samples) * 100 if total_samples > 0 else 0.0
    
    # Affichage des résultats (même style que votre CTCCallback)
    print(f"\n{'='*50}")
    print(f"📊 RÉSULTATS D'ACCURACY (basé sur votre notebook)")
    print(f"{'='*50}")
    print(f"Total échantillons évalués: {total_samples}")
    print(f"Total caractères: {total_characters}")
    print(f"Correspondances exactes: {exact_matches}")
    print(f"")
    print(f"📈 MÉTRIQUES:")
    print(f"  • Character Error Rate (CER): {avg_cer:.2f}%")
    print(f"  • Character Accuracy: {character_accuracy:.2f}%")
    print(f"  • Sequence Accuracy (exact match): {sequence_accuracy:.2f}%")
    print(f"{'='*50}")
    
    # Quelques exemples
    print(f"\n🔍 EXEMPLES DE PRÉDICTIONS:")
    for i in range(min(5, total_samples)):
        true_text = all_original_texts[i]
        pred_text = all_decoded_preds[i]
        match_status = "✅" if true_text == pred_text else "❌"
        print(f"  {i+1}. {match_status}")
        print(f"     Vrai:  '{true_text}'")
        print(f"     Prédit: '{pred_text}'")
        print()
    
    return {
        'cer': avg_cer,
        'character_accuracy': character_accuracy,
        'sequence_accuracy': sequence_accuracy,
        'total_samples': total_samples,
        'exact_matches': exact_matches,
        'total_characters': total_characters,
        'all_true_texts': all_original_texts,
        'all_predicted_texts': all_decoded_preds
    }

def decode_predictions_notebook_style(pred):
    """
    Fonction de décodage identique à celle de votre CTCCallback.
    """
    input_len = np.ones(pred.shape[0]) * pred.shape[1]
    
    # FIXED: Même gestion que votre CTCCallback
    try:
        # Try with merge_repeated (for older versions)
        results = K.ctc_decode(pred, input_length=input_len, greedy=True, merge_repeated=True)[0][0]
    except TypeError:
        # Fall back to without merge_repeated (for newer versions)
        results = K.ctc_decode(pred, input_length=input_len, greedy=True)[0][0]
    
    output_texts = []
    for res in results.numpy():
        output_texts.append(num_to_label(res))
    return output_texts

# =============================================================================
# FONCTION SIMPLE POUR TEST RAPIDE
# =============================================================================

def quick_accuracy_test_notebook_style():
    """
    Test rapide d'accuracy utilisant vos données et modèle existants.
    """
    print("🚀 Test rapide d'accuracy...")
    
    # Test sur quelques batches
    results = calculate_accuracy_from_notebook(
        validation_generator=validation_generator,
        prediction_model=prediction_model,
        num_batches=5  # Testez sur 5 batches pour commencer
    )
    
    return results

# =============================================================================
# FONCTION POUR SAUVEGARDER LES RÉSULTATS
# =============================================================================

def save_accuracy_results(results):
    """
    Sauvegarde les résultats d'accuracy dans /kaggle/working/
    """
    import json
    
    # Préparer les données pour la sauvegarde (sans les listes complètes)
    summary = {
        'cer_percentage': results['cer'],
        'character_accuracy_percentage': results['character_accuracy'],
        'sequence_accuracy_percentage': results['sequence_accuracy'],
        'total_samples_evaluated': results['total_samples'],
        'exact_matches': results['exact_matches'],
        'total_characters': results['total_characters'],
        'evaluation_date': str(pd.Timestamp.now()),
        'alphabet_used': alphabets,
        'max_str_len': max_str_len,
        'num_characters': num_of_characters
    }
    
    # Sauvegarder le résumé
    with open('/kaggle/working/accuracy_results.json', 'w', encoding='utf-8') as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    
    print(f"💾 Résultats sauvegardés dans /kaggle/working/accuracy_results.json")
    print(f"📊 Accuracy finale: {results['character_accuracy']:.2f}%")

# =============================================================================
# UTILISATION
# =============================================================================
"""
# Après votre entraînement, utilisez simplement :

# Test rapide
results = quick_accuracy_test_notebook_style()

# Ou test complet
results = calculate_accuracy_from_notebook(validation_generator, prediction_model)

# Sauvegarder les résultats
save_accuracy_results(results)
"""

'\n# Après votre entraînement, utilisez simplement :\n\n# Test rapide\nresults = quick_accuracy_test_notebook_style()\n\n# Ou test complet\nresults = calculate_accuracy_from_notebook(validation_generator, prediction_model)\n\n# Sauvegarder les résultats\nsave_accuracy_results(results)\n'

In [56]:

# 1. Copiez d'abord toutes les fonctions d'accuracy ci-dessus

# 2. Puis utilisez ce code simple :

print("🎯 CALCUL DE L'ACCURACY DU MODÈLE...")

# Test rapide sur quelques batches
print("\n--- Test rapide (5 batches) ---")
quick_results = quick_accuracy_test_notebook_style()

# Test complet sur toutes les données de validation (optionnel)
print("\n--- Test complet (tous les batches) ---")
# Décommentez la ligne suivante si vous voulez tester sur toutes les données :
# full_results = calculate_accuracy_from_notebook(validation_generator, prediction_model)

# Sauvegarder les résultats
save_accuracy_results(quick_results)

print("\n✅ Évaluation terminée !")

🎯 CALCUL DE L'ACCURACY DU MODÈLE...

--- Test rapide (5 batches) ---
🚀 Test rapide d'accuracy...
Calcul de l'accuracy sur 5 batches...

📊 RÉSULTATS D'ACCURACY (basé sur votre notebook)
Total échantillons évalués: 160
Total caractères: 7001
Correspondances exactes: 27

📈 MÉTRIQUES:
  • Character Error Rate (CER): 11.12%
  • Character Accuracy: 88.88%
  • Sequence Accuracy (exact match): 16.88%

🔍 EXEMPLES DE PRÉDICTIONS:
  1. ❌
     Vrai:  'de chaussettes, soit le double de la commande initiale.'
     Prédit: 'de chaussettes, soit le double de la commande initiale.:'

  2. ❌
     Vrai:  'de saisir la répression des fraudes.'
     Prédit: 'de soinir la répression des frandes.'

  3. ❌
     Vrai:  'J'en désire 20 - CD'
     Prédit: 'M'en désire 20-CD.'

  4. ❌
     Vrai:  'ouvrant un compte chez vous. J'espère'
     Prédit: 'ouvant un compte chrez vrouus. J'espère.'

  5. ❌
     Vrai:  'Je vous ai en effet contacté le mois dernier pour vous demander'
     Prédit: 'Je vous ai en effet cont